# Simulate source changes
Runs inserts, updates and deletes on `sales.customers` in Azure SQL, including several changes of one row and an insert followed by a delete, then waits until CDC has captured them.

In [ ]:
%pip install -q python-tds "pyOpenSSL<24.3" certifi

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("sql_host", "")
dbutils.widgets.text("sql_database", "")
dbutils.widgets.text("sql_user", "")
dbutils.widgets.text("secret_scope", "lab10")
dbutils.widgets.text("secret_key", "azure-sql-password")

from contextlib import closing

import certifi
import pytds


def connect():
    return closing(pytds.connect(
        dsn=dbutils.widgets.get("sql_host"),
        port=1433,
        database=dbutils.widgets.get("sql_database"),
        user=dbutils.widgets.get("sql_user"),
        password=dbutils.secrets.get(dbutils.widgets.get("secret_scope"), dbutils.widgets.get("secret_key")),
        autocommit=True,
        cafile=certifi.where(),
        validate_host=True,
    ))


def insert_rows(cur, table, columns, rows, max_params=2000):
    batch = max(1, max_params // len(columns))
    row_sql = "(" + ", ".join(["%s"] * len(columns)) + ")"
    for i in range(0, len(rows), batch):
        chunk = rows[i:i + batch]
        sql = f"INSERT INTO {table} ({', '.join(columns)}) VALUES " + ", ".join([row_sql] * len(chunk))
        cur.execute(sql, tuple(v for row in chunk for v in row))

In [ ]:
import time

with connect() as conn:
    cur = conn.cursor()
    cur.execute("SELECT CONVERT(varchar(24), sys.fn_cdc_get_max_lsn(), 1)")
    lsn_before = cur.fetchone()[0]
    cur.execute("SELECT MAX(customer_id) FROM sales.customers")
    next_id = cur.fetchone()[0] + 1
    cur.execute("""SELECT TOP 3 customer_id FROM sales.customers c
                   WHERE NOT EXISTS (SELECT 1 FROM sales.orders o WHERE o.customer_id = c.customer_id)
                   ORDER BY customer_id""")
    delete_ids = [r[0] for r in cur.fetchall()]

    new_customers = [
        (next_id + i, f"Customer#NEW{i:03d}", "Building", "Ukraine", f"customer.new{i:03d}@example.com", 5000 + i * 100)
        for i in range(5)
    ]
    insert_rows(cur, "sales.customers", ["customer_id", "name", "segment", "country", "email", "credit_limit"], new_customers)

    cur.execute("UPDATE sales.customers SET credit_limit = credit_limit * 1.10 WHERE customer_id IN (1, 2, 3, 4, 5)")
    cur.execute("UPDATE sales.customers SET segment = 'Machinery' WHERE customer_id = 1")
    cur.execute("UPDATE sales.customers SET email = 'customer.000000002@new-domain.com' WHERE customer_id = 2")
    cur.execute(f"UPDATE sales.customers SET country = 'Poland' WHERE customer_id = {next_id}")
    cur.execute(f"DELETE FROM sales.customers WHERE customer_id IN ({', '.join(map(str, delete_ids))}, {next_id + 4})")

    expected = 5 + 5 + 1 + 1 + 1 + len(delete_ids) + 1
    for _ in range(60):
        cur.execute("SELECT COUNT(*) FROM cdc.sales_customers_CT WHERE __$start_lsn > CONVERT(binary(10), %s, 1) AND __$operation IN (1, 2, 4)", (lsn_before,))
        captured = cur.fetchone()[0]
        if captured >= expected:
            break
        time.sleep(5)

print(f"inserted: {[c[0] for c in new_customers]}")
print(f"updated: [1, 2, 3, 4, 5] (1 and 2 twice), {next_id} after insert")
print(f"deleted: {delete_ids} and {next_id + 4} (inserted in the same batch)")
print(f"change events captured: {captured} of {expected}")
assert captured >= expected, "CDC capture did not catch up"